In [ ]:
# ============================================================
# INSTALL
# ============================================================

!pip install sklearn-genetic-opt -q


# ============================================================
# IMPORT
# ============================================================

import pandas as pd
import numpy as np
import time

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    auc,
    confusion_matrix,
    balanced_accuracy_score,
    matthews_corrcoef
)

from sklearn_genetic import GASearchCV
from sklearn_genetic.space import Integer, Continuous, Categorical

from scipy.stats import t


# ============================================================
# EXPERIMENT NOTES
# ============================================================
# Preprocessing is fit within each outer-training fold via Pipeline.
# The inner 3-fold CV used by GASearchCV sees only the outer-training data.
# SMOTE is not used in this GA-only ablation scenario.
# Baseline preprocessing conventions:
# - Mean imputation for numeric features (including BMI)
# - One-hot encoding: gender, work_type, smoking_status
# - Binary/ordinal encoding: ever_married, Residence_type
# - StandardScaler for LR and SVM numeric features only
# - Same outer folds and base estimator configurations as baseline/oversampling
#
# ============================================================
# LOAD RAW DATA
# ============================================================

df = pd.read_csv(
    '/kaggle/input/datasets/aderahmad/stroke-dataset-raw/healthcare-dataset-stroke-data.csv'
)

# Remove ID
df = df.drop('id', axis=1)

X = df.drop('stroke', axis=1)
y = df['stroke']


# ============================================================
# COLUMN TYPES
# ============================================================

numeric_features = [
    'age',
    'hypertension',
    'heart_disease',
    'avg_glucose_level',
    'bmi'
]

categorical_onehot = [
    'gender',
    'work_type',
    'smoking_status'
]

categorical_ordinal = [
    'ever_married',
    'Residence_type'
]


# ============================================================
# PREPROCESSING
# ============================================================

# ------------------------------------------------------------
# TREE-BASED MODELS
# RF and XGBoost do not require feature scaling
# ------------------------------------------------------------

numeric_transformer_tree = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='mean'))
    ]
)

categorical_onehot_transformer_tree = Pipeline(
    steps=[
        (
            'onehot',
            OneHotEncoder(
                handle_unknown='ignore',
                sparse_output=False
            )
        )
    ]
)

categorical_ordinal_transformer_tree = Pipeline(
    steps=[
        (
            'ordinal',
            OrdinalEncoder(
                categories=[
                    ['No', 'Yes'],
                    ['Rural', 'Urban']
                ],
                handle_unknown='use_encoded_value',
                unknown_value=-1
            )
        )
    ]
)

preprocessor_tree = ColumnTransformer(
    transformers=[
        (
            'num',
            numeric_transformer_tree,
            numeric_features
        ),
        (
            'cat_onehot',
            categorical_onehot_transformer_tree,
            categorical_onehot
        ),
        (
            'cat_ordinal',
            categorical_ordinal_transformer_tree,
            categorical_ordinal
        )
    ],
    remainder='drop'
)


# ------------------------------------------------------------
# LINEAR MODELS
# LR and SVM use StandardScaler
# ------------------------------------------------------------

numeric_transformer_linear = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler())
    ]
)

categorical_onehot_transformer_linear = Pipeline(
    steps=[
        (
            'onehot',
            OneHotEncoder(
                handle_unknown='ignore',
                sparse_output=False
            )
        )
    ]
)

categorical_ordinal_transformer_linear = Pipeline(
    steps=[
        (
            'ordinal',
            OrdinalEncoder(
                categories=[
                    ['No', 'Yes'],
                    ['Rural', 'Urban']
                ],
                handle_unknown='use_encoded_value',
                unknown_value=-1
            )
        )
    ]
)

preprocessor_linear = ColumnTransformer(
    transformers=[
        (
            'num',
            numeric_transformer_linear,
            numeric_features
        ),
        (
            'cat_onehot',
            categorical_onehot_transformer_linear,
            categorical_onehot
        ),
        (
            'cat_ordinal',
            categorical_ordinal_transformer_linear,
            categorical_ordinal
        )
    ],
    remainder='drop'
)


# ============================================================
# BASE MODELS
# ============================================================

rf_model = RandomForestClassifier(
    random_state=42
)

xgb_model = XGBClassifier(
    random_state=42,
    eval_metric='logloss'
)

lr_model = LogisticRegression(
    random_state=42,
    max_iter=1000
)

svm_model = SVC(
    probability=True,
    random_state=42
)


# ============================================================
# HYPERPARAMETER SEARCH SPACE
# ============================================================

# ------------------------------------------------------------
# RANDOM FOREST
# ------------------------------------------------------------

rf_param_grid = {
    'model__n_estimators': Integer(50, 300),
    'model__max_depth': Integer(3, 10),
    'model__min_samples_split': Integer(2, 10),
    'model__min_samples_leaf': Integer(5, 10),
    'model__class_weight': Categorical([
        'balanced',
        'balanced_subsample',
        None
    ])
}



# ------------------------------------------------------------
# XGBOOST
# ------------------------------------------------------------

xgb_param_grid = {
    'model__n_estimators': Integer(50, 300),
    'model__max_depth': Integer(3, 10),
    'model__learning_rate': Continuous(0.01, 0.3),
    'model__subsample': Continuous(0.5, 1.0),
    'model__colsample_bytree': Continuous(0.5, 1.0),
    'model__scale_pos_weight': Continuous(1, 25)
}


# ------------------------------------------------------------
# LOGISTIC REGRESSION
# ------------------------------------------------------------

lr_param_grid = {
    'model__C': Continuous(
        0.01,
        10.0,
        distribution='log-uniform'
    ),
    'model__penalty': Categorical([
        'l2'
    ]),
    'model__solver': Categorical([
        'lbfgs',
        'saga'
    ]),
    'model__class_weight': Categorical([
        None,
        'balanced'
    ])
}


# ------------------------------------------------------------
# SVM
# ------------------------------------------------------------

svm_param_grid = {
    'model__C': Continuous(
        0.01,
        10.0,
        distribution='log-uniform'
    ),
    'model__kernel': Categorical([
        'rbf'
    ]),
    'model__gamma': Categorical([
        'scale',
        'auto'
    ]),
    'model__class_weight': Categorical([
        None,
        'balanced'
    ])
}


# ============================================================
# PIPELINES
# ============================================================

rf_pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor_tree),
        ('model', rf_model)
    ]
)

xgb_pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor_tree),
        ('model', xgb_model)
    ]
)

lr_pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor_linear),
        ('model', lr_model)
    ]
)

svm_pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor_linear),
        ('model', svm_model)
    ]
)


# ============================================================
# MODEL CONFIGURATION
# ============================================================

models = {
    'Random Forest': {
        'pipeline': rf_pipeline,
        'param_grid': rf_param_grid
    },

    'XGBoost': {
        'pipeline': xgb_pipeline,
        'param_grid': xgb_param_grid
    },

    'Logistic Regression': {
        'pipeline': lr_pipeline,
        'param_grid': lr_param_grid
    },

    'SVM': {
        'pipeline': svm_pipeline,
        'param_grid': svm_param_grid
    }
}


# ============================================================
# RESULT STORAGE
# ============================================================

results = {}

best_params = {}

fold_results = []

runtime_results = []


for model_name in models:

    results[model_name] = {
        'accuracy': [],
        'precision': [],
        'recall': [],
        'specificity': [],
        'f1': [],
        'roc_auc': [],
        'average_precision': [],
        'pr_auc': [],
        'balanced_accuracy': [],
        'mcc': []
    }

    best_params[model_name] = []


# ============================================================
# REPEATED STRATIFIED 10-FOLD CROSS-VALIDATION
# ============================================================

cv_outer = RepeatedStratifiedKFold(
    n_splits=10,
    n_repeats=2,
    random_state=42
)


# ============================================================
# OUTER CV
# ============================================================

for fold, (train_idx, test_idx) in enumerate(
    cv_outer.split(X, y),
    start=1
):

    print(
        f"\n{'=' * 60}\n"
        f"OUTER FOLD {fold}\n"
        f"{'=' * 60}"
    )

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]


    # ========================================================
    # EACH MODEL
    # ========================================================

    fold_record = {
        'Fold': fold
    }

    for model_name, config in models.items():

        print(f"\n--- {model_name} ---")

        # ----------------------------------------------------
        # GA SEARCH
        # ----------------------------------------------------

        ga_search = GASearchCV(
            estimator=config['pipeline'],
            cv=3,
            scoring='matthews_corrcoef',
            population_size=10,
            generations=10,
            param_grid=config['param_grid'],
            n_jobs=-1,
            verbose=False
        )

        # ----------------------------------------------------
        # FIT + RUNTIME
        # ----------------------------------------------------

        start_time = time.perf_counter()

        ga_search.fit(
            X_train,
            y_train
        )

        end_time = time.perf_counter()

        runtime = end_time - start_time

        # ----------------------------------------------------
        # BEST MODEL
        # ----------------------------------------------------

        best_model = ga_search.best_estimator_

        best_params[model_name].append(
            ga_search.best_params_
        )

        # ----------------------------------------------------
        # PREDICTION
        # ----------------------------------------------------

        y_pred = best_model.predict(X_test)

        y_prob = best_model.predict_proba(X_test)[:, 1]

        # ----------------------------------------------------
        # CONFUSION MATRIX
        # ----------------------------------------------------

        tn, fp, fn, tp = confusion_matrix(
            y_test,
            y_pred,
            labels=[0, 1]
        ).ravel()

        # ----------------------------------------------------
        # METRICS
        # ----------------------------------------------------

        accuracy = accuracy_score(
            y_test,
            y_pred
        )

        precision = precision_score(
            y_test,
            y_pred,
            zero_division=0
        )

        recall = recall_score(
            y_test,
            y_pred,
            zero_division=0
        )

        specificity = tn / (tn + fp)

        f1 = f1_score(
            y_test,
            y_pred,
            zero_division=0
        )

        roc_auc = roc_auc_score(
            y_test,
            y_prob
        )

        average_precision = average_precision_score(
            y_test,
            y_prob
        )

        pr_precision, pr_recall, _ = precision_recall_curve(
            y_test,
            y_prob
        )

        pr_auc = auc(
            pr_recall,
            pr_precision
        )

        balanced_accuracy = balanced_accuracy_score(
            y_test,
            y_pred
        )

        mcc = matthews_corrcoef(
            y_test,
            y_pred
        )

        # ----------------------------------------------------
        # STORE METRICS
        # ----------------------------------------------------

        results[model_name]['accuracy'].append(
            accuracy
        )

        results[model_name]['precision'].append(
            precision
        )

        results[model_name]['recall'].append(
            recall
        )

        results[model_name]['specificity'].append(
            specificity
        )

        results[model_name]['f1'].append(
            f1
        )

        results[model_name]['roc_auc'].append(
            roc_auc
        )

        results[model_name]['average_precision'].append(
            average_precision
        )

        results[model_name]['pr_auc'].append(
            pr_auc
        )

        results[model_name]['balanced_accuracy'].append(
            balanced_accuracy
        )

        results[model_name]['mcc'].append(
            mcc
        )

        # ----------------------------------------------------
        # FOLD RESULT
        # ----------------------------------------------------

        fold_record[f'{model_name}_accuracy'] = accuracy
        fold_record[f'{model_name}_precision'] = precision
        fold_record[f'{model_name}_recall'] = recall
        fold_record[f'{model_name}_specificity'] = specificity
        fold_record[f'{model_name}_f1'] = f1
        fold_record[f'{model_name}_roc_auc'] = roc_auc
        fold_record[f'{model_name}_average_precision'] = average_precision
        fold_record[f'{model_name}_pr_auc'] = pr_auc
        fold_record[f'{model_name}_balanced_accuracy'] = balanced_accuracy
        fold_record[f'{model_name}_mcc'] = mcc

        fold_record[f'{model_name}_TN'] = tn
        fold_record[f'{model_name}_FP'] = fp
        fold_record[f'{model_name}_FN'] = fn
        fold_record[f'{model_name}_TP'] = tp

        fold_record[f'{model_name}_runtime_seconds'] = runtime

        # ----------------------------------------------------
        # RUNTIME RESULT
        # ----------------------------------------------------

        runtime_results.append({
            'Fold': fold,
            'Model': model_name,
            'Runtime_Seconds': runtime
        })

        # ----------------------------------------------------
        # PRINT RESULT
        # ----------------------------------------------------

        print(
            f"Balanced Accuracy : {balanced_accuracy:.4f}"
        )

        print(
            f"MCC               : {mcc:.4f}"
        )

        print(
            f"PR-AUC            : {pr_auc:.4f}"
        )

        print(
            f"Runtime            : {runtime:.2f} seconds"
        )

        print(
            f"TN={tn}, FP={fp}, FN={fn}, TP={tp}"
        )

    # --------------------------------------------------------
    # SAVE FOLD
    # --------------------------------------------------------

    fold_results.append(fold_record)


# ============================================================
# SUMMARY + 95% CONFIDENCE INTERVAL
# ============================================================

summary_results = []


for model_name, metric_dict in results.items():

    for metric, values in metric_dict.items():

        values = np.array(values)

        n = len(values)

        mean_value = np.mean(values)

        std_value = np.std(
            values,
            ddof=1
        )

        # 95% CI using t-distribution
        t_critical = t.ppf(
            0.975,
            df=n - 1
        )

        margin_error = (
            t_critical
            * std_value
            / np.sqrt(n)
        )

        ci_lower = mean_value - margin_error
        ci_upper = mean_value + margin_error

        summary_results.append({
            'Model': model_name,
            'Metric': metric,
            'Mean': mean_value,
            'Std': std_value,
            'CI95_Lower': ci_lower,
            'CI95_Upper': ci_upper
        })


# ============================================================
# SAVE SUMMARY
# ============================================================

summary_df = pd.DataFrame(
    summary_results
)

summary_df.to_csv(
    'ga_summary_results.csv',
    index=False
)


# ============================================================
# SAVE FOLD RESULTS
# ============================================================

fold_df = pd.DataFrame(
    fold_results
)

fold_df.to_csv(
    'ga_fold_results.csv',
    index=False
)


# ============================================================
# SAVE RUNTIME
# ============================================================

runtime_df = pd.DataFrame(
    runtime_results
)

runtime_df.to_csv(
    'ga_runtime_results.csv',
    index=False
)


# ============================================================
# SAVE BEST PARAMETERS
# ============================================================

for model_name, params in best_params.items():

    params_df = pd.DataFrame(
        params
    )

    params_df.insert(
        0,
        'Fold',
        range(
            1,
            len(params_df) + 1
        )
    )

    filename = (
        model_name
        .lower()
        .replace(' ', '_')
        + '_ga_best_params.csv'
    )

    params_df.to_csv(
        filename,
        index=False
    )


# ============================================================
# PRINT FINAL RESULTS
# ============================================================

print("\n")
print("=" * 70)
print("FINAL GA RESULTS")
print("=" * 70)

for model_name in results:

    print(f"\n{'=' * 60}")
    print(model_name)
    print(f"{'=' * 60}")

    model_summary = summary_df[
        summary_df['Model'] == model_name
    ]

    for _, row in model_summary.iterrows():

        print(
            f"{row['Metric']:20s} "
            f"Mean={row['Mean']:.4f} | "
            f"SD={row['Std']:.4f} | "
            f"95% CI="
            f"[{row['CI95_Lower']:.4f}, "
            f"{row['CI95_Upper']:.4f}]"
        )

print("\nAll results saved.")